# Class activation maps

In [254]:
import torch
from torchvision.transforms import transforms
import numpy as np
import ants
import os
import pandas as pd
import pandas as pd 
import numpy as np
import nibabel as nib
import ants
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
plotly.tools.set_credentials_file(username='Qqin', api_key='GFe6E0uLlZmHtoobrxiv')
import plotly.plotly as py
import plotly.graph_objs as go
from data_loader import NiftyLoader
from torch.utils import data
from matplotlib.backends.backend_pdf import PdfPages
plt.rc(group={'font': {'size': 'larger'}})
sns.set("paper", font_scale=3)
sns.set_style("ticks", {"xtick.major.size": 0,
                        "ytick.major.size": 0,
                        'font.family': ['sans-serif']})
%matplotlib inline

In [29]:
from inceptionv3 import InceptionAttention3D
#path = '/data1/qinqian/MRI/01_PVL_AI/scripts/models/pytorch/results/models_Leukomalacia/Leukomalacia_modal0,1,2_class_models_initiallr0.000170_batch32_drop0.0/params5_dataiter_25_currentlr_0.00017_validloss_0.01373_validacc_0.82249_train_loss_0.01394_train_acc_0.80432.ckpt'
model = InceptionAttention3D(1, drop=0.5)
path = '/data1/qinqian/MRI/01_PVL_AI/scripts/models/pytorch/results/models_Leukomalacia_pure/Leukomalacia_modal0_class_models_initiallr0.000170_batch32_drop0.2/params64_dataiter_16_currentlr_6.294407596289999e-06_validloss_0.01331_validacc_0.88636_train_loss_0.00012_train_acc_1.0.ckpt'
model.load_state_dict(torch.load(path)['model'])
#model = model.to('cuda')
model.eval()

InceptionAttention3D(
  (features): Sequential(
    (0): Conv3d(1, 64, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
    (1): ELU(alpha=1.0)
    (2): MaxPool3d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Inception(
      (p1): Conv3dBlock(
        (block): Sequential(
          (0): Conv3d(64, 64, kernel_size=(1, 1, 1), stride=(1, 1, 1))
          (1): ELU(alpha=1.0)
        )
      )
      (p2_1): Conv3dBlock(
        (block): Sequential(
          (0): Conv3d(64, 96, kernel_size=(1, 1, 1), stride=(1, 1, 1))
          (1): ELU(alpha=1.0)
        )
      )
      (p2_2): Conv3dBlock(
        (block): Sequential(
          (0): Conv3d(96, 128, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
          (1): ELU(alpha=1.0)
        )
      )
      (p3_1): Conv3dBlock(
        (block): Sequential(
          (0): Conv3d(64, 16, kernel_size=(1, 1, 1), stride=(1, 1, 1))
          (1): ELU(alpha=1.0)
        )
      )
      (p3_2): Conv3dBlo

In [30]:
test_bids = pd.read_csv('results/seed1_test.csv', index_col=0)
train_bids = pd.read_csv('results/seed1_train.csv', index_col=0)
valid_bids = pd.read_csv('results/seed1_valid.csv', index_col=0)

In [31]:
np.intersect1d(test_bids.index, train_bids.index)

array([], dtype=int64)

In [32]:
np.intersect1d(test_bids.index, valid_bids.index)

array([], dtype=int64)

In [39]:
test_metrics = pd.read_csv('/data1/qinqian/MRI/01_PVL_AI/scripts/models/pytorch/results/models_Leukomalacia_pure/Leukomalacia_modal0_class_models_initiallr0.000170_batch32_drop0.2_test.csv')

In [37]:
seed = 1
np.random.seed(seed)
torch.backends.cudnn.benchmark = True
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

nifty = NiftyLoader(test_bids, 
                    [0], 'Leukomalacia', 635)
#nifty_loader = data.DataLoader(nifty, batch_size=8, num_workers=10)
results = []
for i, j in nifty:
    results.append(j)

In [42]:
(test_metrics.predict==np.array(results)[:, 1]).mean()

0.8407643312101911

In [43]:
from sklearn.metrics import roc_auc_score
roc_auc_score(test_metrics.labels, test_metrics.outputs)

0.9078062214998419

In [44]:
true_pos,  = np.where(test_metrics.predict==np.array(results)[:, 1])

In [45]:
np.where(test_metrics.iloc[true_pos, ].predict==1)

(array([  1,   7,  12,  14,  16,  23,  24,  30,  32,  33,  35,  36,  38,
         39,  41,  42,  46,  47,  48,  52,  57,  70,  71,  72,  73,  74,
         76,  77,  84,  87,  89,  97, 103, 112, 115, 122, 125, 128, 137,
        146, 149, 151, 152, 154, 157, 164, 168, 169, 170, 171, 172, 173,
        174, 180, 181, 182, 183, 192, 196, 197, 207, 208, 209, 213, 215,
        218, 221, 222, 224, 233, 235, 236, 242, 243, 244, 246, 247, 249,
        250, 251, 252, 253, 255, 260]),)

In [218]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(test_metrics.loc[:, 'labels'], test_metrics.loc[:, 'predict']))
sigmoid = torch.nn.Sigmoid()

[[180  27]
 [ 23  84]]


In [343]:
def draw_cam(index, pdf, slices_range=range(52, 58)):
    input_id = test_bids.index[true_pos][index]
    t2f = ants.image_read(f'results/processed/{input_id}_t2f_processed.nii.gz')
    IMAGES_MEAN_SD = {0: [0.5872402, 0.2024262],
                      1: [0.5821896, 0.1871165],
                      2: [0.4300238, 0.2120626],
                      3: [0.4056657, 0.1636801],
                      4: [0.3866455, 0.133584]}

    MACHINE_INDEX = {'SIEMENS-Avanto-1.5T': 0,
                     'GE MEDICAL SYSTEMS-DISCOVERY MR750-3.0T': 1,
                     'SIEMENS-Skyra-3.0T': 2}

    def preprocess(image, modal):
        return (image - IMAGES_MEAN_SD[modal][0])/IMAGES_MEAN_SD[modal][1]

    features = []
    def hook_feature(module, input, output):
        features.append(output.data.cpu().numpy())
    model._modules.get('features')[-3].register_forward_hook(hook_feature)
    model._modules.get('features').register_forward_hook(hook_feature)
    prediction = model(torch.from_numpy(np.expand_dims(nifty[true_pos[index]][0], 0)))
    h_x = sigmoid(prediction).data.squeeze()
    probs, idx = h_x.sort(0, True)
    probs = probs.numpy()
    idx = idx.numpy()
    params = list(model.parameters())
    weight = params[-2].cpu().detach().numpy()

    bz, nc, x, y, z = features[0].shape
    cam = weight[idx[0]].dot(features[0].reshape((nc, x*y*z)))
    output_cam = cam.reshape(x, y, z)
#    output_cam = output_cam / np.max(output_cam)
    output_cam = (output_cam-np.min(output_cam)) / np.max(output_cam)    
#    output_cam[output_cam<0.4] = 0
    t2f_down = t2f.resample_image(output_cam.shape, use_voxels=True)
    t2f_cam = ants.make_image(t2f_down.shape, voxval=output_cam, 
                    spacing=ants.get_spacing(t2f_down), 
                    direction=ants.get_direction(t2f_down), 
                    origin=ants.get_origin(t2f_down))
    t2f_cam = t2f_cam.resample_image(t2f.shape, use_voxels=True, interp_type=2)
    t2f_cam = t2f_cam * ants.threshold_image(t2f_cam, np.quantile(t2f_cam.numpy(), 0.8), 1)
    slices = []
    slices_heat = []
    for s in slices_range:
        slices.append(np.rot90(t2f[:, :, s])[:, ::-1])
        slices_heat.append(np.rot90(t2f_cam[:, :, s])[:, ::-1])    

    print(np.hstack(slices).shape)
    fig = plt.figure(figsize=(24, 10))
#    vmin, vmax = t2f.quantile((0.05, 0.95))
    ax = fig.add_axes([0.02, 0, 0.9, 0.9])
    ax = ax.imshow(np.hstack(slices), cmap=plt.cm.gray, interpolation='nearest', 
                    vmin=0, vmax=1)
    im = plt.imshow(np.hstack(slices_heat), cmap=plt.cm.RdBu_r, alpha=0.46)
    plt.axis('off')
    cax = fig.add_axes([0.05, 0.05, 0.25, 0.04])
    plt.colorbar(im, orientation='horizontal', cax=cax, ax=im)
    plt.title(index)
    #(t2f).plot(axis=2, slices=slices_range, ncol=len(slices_range), figsize=5)
    #np.quantile(output_cam.squeeze(), 0.9), np.std(output_cam.squeeze()), np.min(output_cam.squeeze()), np.max(output_cam.squeeze())
    pdf.savefig()

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

candidates, = np.where(test_metrics.iloc[true_pos, ].predict==1)
with PdfPages('results/Fig3_cam.pdf') as pdf:
    for c in candidates[1:]:
        fig = draw_cam(c, pdf)

(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)


/home/qinqian/.local/lib/python3.6/site-packages/ipykernel_launcher.py:50: RuntimeWarning:

More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).



(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
(112, 522)
